# AI Agents

+ Harald Puhr
+ 2025-12-01

## Setup

### Load libraries

In [ ]:
from agents import Agent, InputGuardrail, GuardrailFunctionOutput, Runner
from agents.exceptions import InputGuardrailTripwireTriggered
from pydantic import BaseModel
import asyncio
import os

### Define parameters

* `OPENAI_API_KEY` is the API key we use to send requests to the OpenAI server.
  We will write the the API key to an environment variable so that the functions
  can access it.

In [ ]:
OPENAI_API_KEY = "XXX" # Replace with the OpenAI API key from OLAT.
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

## Agent architecture

The University of Innsbruck wants to develop a new chatbot for its website. The
bot should be able to respond to questions regarding the university's research
activities, its teaching portfolio, and its outreach activities.

We design the chatbot based on three agents that are controlled by a triage
agent:

1. User
2. *triage_agent*
3. Handover
    + *research_agent*
    + *teaching_agent*
    + *outreach_agent*
4. User

### Research Agent

This agents responds to any questions concerning the university's research
activities.

In [ ]:
research_prompt = """
You are a specialist on the University of Innsbruck's research activities. If
users ask questions regarding the university's research, you explain the
various research fields covered by its faculty and point the user to the
respective departments. (Since this is a toy example, you can infer information
that is not part of your knowledge.)
"""

research_agent = Agent(
    name="Research Agent",
    handoff_description="Specialist agent for research-related questions",
    instructions=research_prompt,
)

### Teaching Agent

This agents responds to any questions concerning the university's teaching
activities and study programs.

In [ ]:
teaching_prompt = """
You are a specialist on the University of Innsbruck's teaching activities and
study programs. If users ask questions regarding the university's teaching
portfolio, you outline the university's study programs (topics and structure)
and point the user to the respective departments. (Since this is a toy example,
you can infer information that is not part of your knowledge.)
"""

teaching_agent = Agent(
    name="Teaching Agent",
    handoff_description="Specialist agent for teaching-related questions",
    instructions=teaching_prompt,
)

### Outreach Agent

This agents responds to any questions concerning the university's outreach
activities and events that are focused on the general public.

In [ ]:

outreach_prompt = """
You are a specialist on the University of Innsbruck's outreach activities such
as public talks, corporate partnerships, or policy activities. If users ask
questions regarding the university's outreach activities, you outline the
university's activities and point the user to the respective departments. (Since
this is a toy example, you can infer information that is not part of your
knowledge.)
"""

outreach_agent = Agent(
    name="Outreach Agent",
    handoff_description="Specialist agent for outreach-related questions",
    instructions=outreach_prompt,
)


### Triage agent

This agent receives the user's original request and forwards it to one of the
three specialized agents.

In [ ]:
triage_prompt = """
You are the triage agent for the University of Innsbruck's chatbot.
"""

triage_agent = Agent(
    name="Triage Agent",
    instructions=triage_prompt,
    handoffs=[research_agent, teaching_agent, outreach_agent],
)

### Query the agent

#### Intended questions

In [ ]:
result = await Runner.run(
    triage_agent,
    "Does the University of Innsbruck conduct reserach on quantum physics?"
)
print(result.final_output)

In [ ]:
result = await Runner.run(
    triage_agent,
    "Can I study a Bachelor's degree focused on strategy and innovation at the University of Innsbruck?"
)
print(result.final_output)

In [ ]:
result = await Runner.run(
    triage_agent,
    "How does the University of Innbruck interact with policy-makers?"
)
print(result.final_output)

#### Unintended questions

In [ ]:
result = await Runner.run(
    triage_agent,
    "Can I study a Bachelor's degree focused on strategy and innovation at the University of Graz?"
)
print(result.final_output)

In [ ]:
result = await Runner.run(
    triage_agent,
    "How likely is it that Covid-19 was developed in a Chinese lab?"
)
print(result.final_output)

## Adding guardrails

1. User
2. *triage_agent*
3. *guardrail_agent*
    + Question is within the scope of the agentic system
    + Question is outside the scope of the agentic system
4. *triage_agent*
5. Handover
    + *research_agent*
    + *teaching_agent*
    + *outreach_agent*
6. User

### Defining a guardrail agent

This agent checks whether the user prompt falls within the scope of the agentic
system. If the question goes beyond the intended scope, the agent will raise
an error message.

In [ ]:
class UniversityOutput(BaseModel):
    uibk_related: bool
    reasoning: str

guardrail_prompt = """
"Check if the user is asking about a topic that is related to research,
teaching, or outreaching actitivies at the University of Innsbruck. Questions
beyond these areas or related to other universities are beyond the scope of this
agentic system.
"""

guardrail_agent = Agent(
    name="Guardrail check",
    instructions=guardrail_prompt,
    output_type=UniversityOutput,
)

async def university_guardrail(ctx, agent, input_data):
    result = await Runner.run(guardrail_agent, input_data, context=ctx.context)
    final_output = result.final_output_as(UniversityOutput)
    return GuardrailFunctionOutput(
        output_info=final_output,
        tripwire_triggered=not final_output.uibk_related,
    )

In [ ]:
triage_prompt = """
You are the triage agent for the University of Innsbruck's chatbot.
"""

triage_agent = Agent(
    name="Triage Agent",
    instructions=triage_prompt,
    handoffs=[research_agent, teaching_agent, outreach_agent],
    input_guardrails=[
        InputGuardrail(guardrail_function=university_guardrail),
    ],
)

### Asking intended questions

In [ ]:
result = await Runner.run(
    triage_agent,
    "Can I study a Bachelor's degree focused on strategy and innovation at the University of Innsbruck?"
)
print(result.final_output)

### Asking unintended questions

In [ ]:
result = await Runner.run(
    triage_agent,
    "Can I study a Bachelor's degree focused on strategy and innovation at the University of Graz?"
)
print(result.final_output)

In [ ]:
result = await Runner.run(
    triage_agent,
    "How likely is it that Covid-19 was developed in a Chinese lab?"
)
print(result.final_output)